In [ ]:
import os
import rioxarray as rioxr
import xarray as xr
from dataclasses import asdict
from dscreator.cfarray.attributes import DatasetAttrsGrid, VariableAttrs, LongitudeAttrs, LatitudeAttrs
from dscreator import utils
from dscreator.storage import get_storage_handler
from dscreator.config import SETTINGS


def load_urchin_raster(path, name, long_name):
    da = rioxr.open_rasterio(path, masked=False).drop_vars("band")
    da = da.squeeze("band", drop=True)
    da.name = name
    da.attrs = asdict(VariableAttrs(long_name=long_name, units="g m-2", short_name=name))
    da.x.attrs = asdict(LongitudeAttrs())
    da.y.attrs = asdict(LatitudeAttrs())
    da = da.drop_vars("spatial_ref")
    return da


# Full study area layer
full_da = load_urchin_raster(
    "Lavin_et_al_S_droebachiensis_final_biomass_sqm_wgs84_RF_GLS_XY_September.tif",
    name="Predicted_urchin_biomass",
    long_name="Strongylocentrotus droebachiensis predicted biomass",
)

# Masked layer (conservative MESS >= 0 prediction range), same grid as the full layer
masked_da = load_urchin_raster(
    "Lavin_et_al_S_droebachiensis_final_biomass_sqm_MESS_ge0_wgs84_RF_GLS_XY_September.tif",
    name="Predicted_urchin_biomass_masked",
    long_name="Strongylocentrotus droebachiensis predicted biomass (masked to a conservative prediction range, MESS >= 0)",
)

In [ ]:
ds = xr.merge([full_da, masked_da])
ds.attrs = asdict(
    DatasetAttrsGrid(
        title="Strongylocentrotus droebachiensis predicted biomass (g/m2)",
        title_no="Strongylocentrotus droebachiensis predicted biomass (g/m2)",
        summary=(
            "Predicted biomass of Strongylocentrotus droebachiensis from the combined products of "
            "a geospatial Random Forest model and an ensemble Species Distribution Model, from the "
            "manuscript: (Insert link here when available). The dataset contains two layers: predictions "
            "across the full study area (Predicted_urchin_biomass) and predictions masked to a conservative "
            "prediction range (MESS >= 0) across the study area (Predicted_urchin_biomass_masked)."
        ),
        summary_no="Strongylocentrotus droebachiensis predicted biomass (g/m2)",
        creator_email="charles.lavin@niva.no",
        creator_name="Charles P. Lavin",
        creator_institution="Norsk institutt for vannforskning",
        creator_type="person",
        # https://gcmd.earthdata.nasa.gov/KeywordViewer
        keywords=",".join(
            [
                "GCMDSK:Science Keywords > EARTH SCIENCE > BIOLOGICAL CLASSIFICATION > ANIMALS/INVERTEBRATES > ECHINODERMS > SEA URCHINS",
                "GCMDLOC:CONTINENT > EUROPE > NORTHERN EUROPE > SCANDINAVIA > NORWAY",
            ]
        ),
        keywords_vocabulary=",".join(
            [
                "GCMDSK:GCMD Science Keywords:https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/sciencekeywords",
                "GCMDLOC:GCMD Locations:https://gcmd.earthdata.nasa.gov/kms/concepts/concept_scheme/locations",
            ]
        ),
        time_coverage_end="",
        time_coverage_start="",
        iso_topic_category="environment",
        spatial_representation="grid",
        date_created="2025-12-01T12:00:00Z",
        project="230189 - Removal and utilization of sea urchin barrens to restore macroalgae forests, eco",
        geospatial_lon_min=float(ds.x.min()),
        geospatial_lon_max=float(ds.x.max()),
        geospatial_lat_min=float(ds.y.min()),
        geospatial_lat_max=float(ds.y.max()),
        history="",
    )
)
ds.attrs["id"] = "522d5685-7157-407c-bb22-6a882ed5cbd8"

In [ ]:
SETTINGS.storage_path = os.path.join(os.getcwd(), "..", "catalog")
SETTINGS.storage_path = "gs://nivaprod-1-senda"

ds["x"] = ds["x"].astype("float32")
ds["y"] = ds["y"].astype("float32")
# For transfer directly into the test thredds server use

get_storage_handler(
    grouping="refarktis",
    dataset_name="S_droebachiensis_final_biomass_sqm_wgs84",
    filename_prefix="area-density",
).save_dataset(ds)